In [ ]:
!pip install transformers x-transformers sentencepiece --quiet

In [ ]:
import re
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from transformers import AutoTokenizer, AutoModel
from x_transformers import Decoder


In [ ]:
BATCH_SIZE = 16
LR = 2e-5
EPOCHS = 5
MAX_LEN = 256

DECODER_DIM = 768
DECODER_DEPTH = 2
DECODER_HEADS = 8

labels = [
    "Happiness",
    "Sadness",
    "Anger",
    "Disgust",
    "Surprise",
    "Fear"
]

NUM_CLASSES = len(labels)

In [ ]:
MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
df = pd.read_csv("/kaggle/input/sied-thai/SIED-Thai.csv")

df = df[['Tweets'] + labels].dropna()

# ❗ baseline: ไม่ลบ sample ว่าง (ให้ model เรียนเอง)
# df = df[df[labels].sum(axis=1) > 0]

In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df['text'] = df['Tweets'].apply(clean_text)

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2)
train_df, val_df  = train_test_split(train_df, test_size=0.1)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [ ]:
class EmotionDataset(Dataset):

    def __init__(self, df):
        self.texts = df['text'].tolist()
        self.labels = df[labels].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return (
            encoding["input_ids"].squeeze(0),
            encoding["attention_mask"].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

train_loader = DataLoader(EmotionDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(EmotionDataset(val_df), batch_size=BATCH_SIZE)
test_loader  = DataLoader(EmotionDataset(test_df), batch_size=BATCH_SIZE)

In [ ]:
class Model(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(MODEL_NAME)

        self.decoder = Decoder(
            dim=DECODER_DIM,
            depth=DECODER_DEPTH,
            heads=DECODER_HEADS
        )

        self.fc = nn.Linear(DECODER_DIM, NUM_CLASSES)

    def forward(self, input_ids, attention_mask):

        x = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state

        x = self.decoder(
            x,
            mask=attention_mask.bool()
        )

        # mean pooling
        mask = attention_mask.unsqueeze(-1).float()
        x = (x * mask).sum(1) / mask.sum(1)

        return self.fc(x)

model = Model().to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# ❗ baseline: no class weighting
criterion = nn.BCEWithLogitsLoss()

In [ ]:
@torch.no_grad()
def evaluate(loader):

    model.eval()

    preds = []
    targets = []

    for input_ids, attention_mask, y in loader:

        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        logits = model(input_ids, attention_mask)

        probs = torch.sigmoid(logits)

        # ❗ baseline threshold = 0.5
        pred = (probs > 0.5).int().cpu().numpy()

        preds.append(pred)
        targets.append(y.numpy())

    y_pred = np.vstack(preds)
    y_true = np.vstack(targets)

    f1_micro = f1_score(y_true, y_pred, average="micro")
    f1_macro = f1_score(y_true, y_pred, average="macro")

    return f1_micro, f1_macro

In [ ]:
for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for input_ids, attention_mask, y in train_loader:

        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)

        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    f1_micro, f1_macro = evaluate(val_loader)

    print(
        f"Epoch {epoch+1} | "
        f"loss={total_loss:.4f} | "
        f"f1_micro={f1_micro:.4f} | "
        f"f1_macro={f1_macro:.4f}"
    )

In [ ]:
f1_micro, f1_macro = evaluate(test_loader)

print("\nFINAL TEST")
print("F1 Micro :", f1_micro)
print("F1 Macro :", f1_macro)